# PodcastGen-Agent
Generate a complete podcast MP3 from any topic.

Works on **Google Colab** and **Kaggle**.

## Before you start

### Google Colab
1. Select **Runtime → Change runtime type → T4 GPU**.
2. Run cells **1 and 2** before any other cell (clone repo + install deps).

### Kaggle
1. Open **Notebook → Settings → Accelerator → GPU T4 x2** (or GPU P100).
2. Turn **Internet** ON (required for `git clone` and `pip install`).
3. Run cells **1 and 2** before any other cell.

### Both platforms
1. Cell 1 must pull the latest GitHub code before you generate.
2. Start with `DURATION = 2` (about 10 dialogue lines, roughly 10 to 20 minutes on T4).
3. If the session disconnects, copy the `run_id` from the logs into `RUN_ID` and rerun cell 4.
4. Playback/download cells (5 and 6) work after generation even if you skip rerunning cell 4, but cells 1 and 2 must have been run once in this session.


In [ ]:
# Clone or update the repository first (Colab or Kaggle).
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Rashal10/PodcastGen-Agent.git"


def _detect_notebook_env() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or Path("/content").exists():
        return "colab"
    return "local"


ENV_NAME = _detect_notebook_env()
if ENV_NAME == "kaggle":
    WORK_ROOT = Path("/kaggle/working")
elif ENV_NAME == "colab":
    WORK_ROOT = Path("/content")
else:
    WORK_ROOT = Path.cwd()

REPO_DIR = WORK_ROOT / "PodcastGen-Agent"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
else:
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])

os.chdir(REPO_DIR)
os.environ["PODCAST_GEN_REPO"] = str(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print(f"Environment: {ENV_NAME}")
print("Repository ready:", os.getcwd())
print("Package path ready:", (REPO_DIR / "podcast_gen_agent").is_dir())


In [ ]:
# Install notebook-compatible dependencies. This intentionally does not install
# Audiocraft because Audiocraft 1.3 requires Python 3.9 and torch 2.1.
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
if not (REPO_DIR / "requirements-colab.txt").is_file():
    raise RuntimeError(f"Repository not ready at {REPO_DIR}. Run cell 1 first.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


def _ensure_system_packages() -> None:
    if shutil.which("ffmpeg") and shutil.which("espeak-ng"):
        print("ffmpeg and espeak-ng already installed")
        return

    packages = ["ffmpeg", "espeak-ng"]
    last_error: Exception | None = None
    for prefix in ([], ["sudo"]):
        try:
            subprocess.check_call(prefix + ["apt-get", "update", "-qq"])
            subprocess.check_call(prefix + ["apt-get", "install", "-y", "-qq", *packages])
            print("Installed system packages:", ", ".join(packages))
            return
        except Exception as exc:  # noqa: BLE001
            last_error = exc
    raise RuntimeError(
        "Could not install ffmpeg/espeak-ng. Install them manually, then rerun."
    ) from last_error


_ensure_system_packages()

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "uninstall",
        "-y",
        "TTS",
        "audiocraft",
        "langchain",
        "langchain-community",
        "langchain-text-splitters",
    ],
    check=False,
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements-colab.txt",
    ]
)
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-e",
        ".",
        "--no-deps",
    ]
)

# Verify imports in a fresh Python subprocess so cached notebook modules cannot hide
# a broken installation.
verification = r"""
import shutil
import torch
import tokenizers
import transformers
from packaging import version
from podcast_gen_agent.compat import ensure_langchain_compat, ensure_transformers_compat

ensure_transformers_compat()
ensure_langchain_compat()

from langchain_core.globals import get_debug
from TTS.api import TTS
from transformers import MusicgenForConditionalGeneration
from podcast_gen_agent.graph import get_graph

assert torch.cuda.is_available(), "GPU runtime is not enabled"
assert shutil.which("ffmpeg"), "ffmpeg is missing"
assert shutil.which("espeak-ng"), "espeak-ng is missing"
assert version.parse(transformers.__version__) >= version.parse("4.57.5"), transformers.__version__
assert version.parse(tokenizers.__version__) >= version.parse("0.22.0"), tokenizers.__version__
assert get_debug() in (True, False)
get_graph()
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("tokenizers:", tokenizers.__version__)
print("XTTS import: OK")
print("MusicGen import: OK")
print("LangGraph compile: OK")
print("ffmpeg and eSpeak: OK")
"""
subprocess.run([sys.executable, "-c", verification], check=True, cwd=str(REPO_DIR))

import podcast_gen_agent

env = (
    "kaggle"
    if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists())
    else "colab"
    if (os.environ.get("COLAB_RELEASE_TAG") or Path("/content").exists())
    else "local"
)
print("All dependencies and pipeline imports are ready.")
print("Kernel package path:", podcast_gen_agent.__file__)
print("Environment:", env)


In [ ]:
import os
from pathlib import Path

import torch

env = "kaggle" if (os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists()) else (
    "colab" if (os.environ.get("COLAB_RELEASE_TAG") or Path("/content").exists()) else "local"
)
gpu_hint = (
    "Settings → Accelerator → GPU T4 x2 (or GPU P100), and turn Internet ON."
    if env == "kaggle"
    else "Runtime → Change runtime type → T4 GPU."
)

assert torch.cuda.is_available(), (
    f"No GPU detected ({env}). Enable a GPU ({gpu_hint}) "
    "then run the notebook again from cell 1."
)
print(f"Environment: {env}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# Change only TOPIC and DURATION for each podcast.
import os
import sys
import json
from pathlib import Path

def _detect_notebook_env() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or Path("/content").exists():
        return "colab"
    return "local"


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        Path("/kaggle/working/PodcastGen-Agent"),
        Path("/content/PodcastGen-Agent"),
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(Path("/content/PodcastGen-Agent/outputs"))
    add(Path("/kaggle/working/PodcastGen-Agent/outputs"))
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if content_root.exists():
            for manifest in content_root.rglob("run_manifest.json"):
                add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    resolved = path.resolve()
    env = _detect_notebook_env()
    if env == "colab":
        from google.colab import files

        files.download(str(resolved))
        return
    if env == "kaggle":
        from IPython.display import FileLink, display

        dest = Path("/kaggle/working") / resolved.name
        if resolved != dest.resolve():
            dest.write_bytes(resolved.read_bytes())
            print(f"Copied to {dest}")
        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download from the Kaggle Output panel.")
        return
    print(f"File ready at: {resolved}")

REPO_DIR = _prepare_repo()
ENV_NAME = _detect_notebook_env()

TOPIC = "LoRA and QLoRA"
DURATION = 2
RUN_ID = ""

assert TOPIC.strip(), "TOPIC must not be empty"
assert 1 <= DURATION <= 30, "DURATION must be between 1 and 30 minutes"

os.environ["COQUI_TOS_AGREED"] = "1"
os.environ["PODCAST_LOG_LEVEL"] = "INFO"
os.environ["PODCAST_MIN_GPU_FREE_MB"] = "128"
os.environ["PODCAST_MAX_SCRIPT_LINES"] = "12"

# Voices: better default presets (Andrew Chipper + Ana Florence)
# os.environ["PODCAST_HOST_VOICE"] = "Craig Gutsy"
# os.environ["PODCAST_GUEST_VOICE"] = "Gracie Wise"

# Optional voice cloning (6 to 30 second WAV files)
# os.environ["PODCAST_HOST_SPEAKER_WAV"] = str(REPO_DIR / "voices" / "host_ref.wav")
# os.environ["PODCAST_GUEST_SPEAKER_WAV"] = str(REPO_DIR / "voices" / "guest_ref.wav")

# Research: Wikipedia + arXiv by default; add API keys for stronger web search
# os.environ["PODCAST_RESEARCH_PROVIDERS"] = "wikipedia,arxiv,tavily,brave,duckduckgo"
# os.environ["PODCAST_TAVILY_API_KEY"] = "tvly-xxxxxxxx"
# os.environ["PODCAST_BRAVE_API_KEY"] = "BSA-xxxxxxxx"

argv = ["podcast_gen_agent.main", TOPIC, "--duration", str(DURATION)]
if RUN_ID.strip():
    argv.extend(["--resume", RUN_ID.strip()])
sys.argv = argv

print("Environment:", ENV_NAME)
print("Repo:", REPO_DIR)
print("Running:", " ".join(argv[1:]))

try:
    from podcast_gen_agent.compat import ensure_langchain_compat
    from podcast_gen_agent.main import main

    ensure_langchain_compat()
    main()
except SystemExit as exc:
    if exc.code not in (0, None):
        from podcast_gen_agent.config import settings

        outputs = settings.output_dir.resolve()
        manifests = sorted(outputs.glob("*/run_manifest.json"), key=lambda p: p.stat().st_mtime)
        if manifests:
            manifest = json.loads(manifests[-1].read_text(encoding="utf-8"))
            print("Latest manifest:", manifests[-1])
            print("Run id:", manifest.get("run_id"))
            print("Status:", manifest.get("status"))
            print("Error:", manifest.get("error"))
        raise

latest = _find_latest_podcast_mp3()
assert latest is not None and latest.stat().st_size > 0, (
    "Generation finished without producing a non-empty MP3. Check the logs above."
)
print(f"SUCCESS: {latest} ({latest.stat().st_size / 1_000_000:.1f} MB)")


In [ ]:
import os
import sys
from pathlib import Path

from IPython.display import Audio, display

def _detect_notebook_env() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or Path("/content").exists():
        return "colab"
    return "local"


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        Path("/kaggle/working/PodcastGen-Agent"),
        Path("/content/PodcastGen-Agent"),
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(Path("/content/PodcastGen-Agent/outputs"))
    add(Path("/kaggle/working/PodcastGen-Agent/outputs"))
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if content_root.exists():
            for manifest in content_root.rglob("run_manifest.json"):
                add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    resolved = path.resolve()
    env = _detect_notebook_env()
    if env == "colab":
        from google.colab import files

        files.download(str(resolved))
        return
    if env == "kaggle":
        from IPython.display import FileLink, display

        dest = Path("/kaggle/working") / resolved.name
        if resolved != dest.resolve():
            dest.write_bytes(resolved.read_bytes())
            print(f"Copied to {dest}")
        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download from the Kaggle Output panel.")
        return
    print(f"File ready at: {resolved}")

REPO_DIR = _prepare_repo()
latest = _find_latest_podcast_mp3()
if latest:
    size_mb = latest.stat().st_size / 1_000_000
    print(f"Playing: {latest} ({size_mb:.1f} MB)")
    # Notebooks need embedded bytes; a file path often fails for multi-MB MP3s.
    display(Audio(data=latest.read_bytes()))
else:
    print("No podcast mp3 found.")
    print("cwd:", os.getcwd())
    print("Repo:", REPO_DIR)
    print(
        "If nothing was generated, the runtime may have wiped working storage. "
        "Re-run cells 1, 2, and 4."
    )


In [ ]:
import os
import sys
from pathlib import Path

def _detect_notebook_env() -> str:
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle/working").exists():
        return "kaggle"
    if os.environ.get("COLAB_RELEASE_TAG") or Path("/content").exists():
        return "colab"
    return "local"


def _resolve_repo_dir() -> Path:
    env_repo = os.environ.get("PODCAST_GEN_REPO")
    if env_repo:
        candidate = Path(env_repo).resolve()
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate
    for candidate in (
        Path("/kaggle/working/PodcastGen-Agent"),
        Path("/content/PodcastGen-Agent"),
        Path.cwd(),
        Path.cwd().parent,
    ):
        if (candidate / "podcast_gen_agent").is_dir():
            return candidate.resolve()
    raise RuntimeError("Repository not found. Run cell 1 first.")


def _prepare_repo(repo_dir: Path | None = None) -> Path:
    root = _resolve_repo_dir() if repo_dir is None else Path(repo_dir).resolve()
    os.chdir(root)
    os.environ["PODCAST_GEN_REPO"] = str(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    return root


def _find_latest_podcast_mp3() -> Path | None:
    roots: list[Path] = []
    seen: set[str] = set()

    def add(path: Path | str) -> None:
        resolved = Path(path).resolve()
        key = str(resolved)
        if key in seen:
            return
        seen.add(key)
        roots.append(resolved)

    repo = Path(os.environ.get("PODCAST_GEN_REPO", Path.cwd())).resolve()
    add(repo / "outputs")
    add(Path.cwd() / "outputs")
    add(Path("/content/PodcastGen-Agent/outputs"))
    add(Path("/kaggle/working/PodcastGen-Agent/outputs"))
    for content_root in (Path("/content"), Path("/kaggle/working")):
        if content_root.exists():
            for manifest in content_root.rglob("run_manifest.json"):
                add(manifest.parent)

    best: Path | None = None
    best_mtime = -1.0
    for root in roots:
        if not root.exists():
            continue
        mp3_files = [p for p in root.rglob("podcast_*.mp3") if p.stat().st_size > 0]
        if not mp3_files:
            mp3_files = [p for p in root.rglob("*.mp3") if p.stat().st_size > 0]
        for path in mp3_files:
            mtime = path.stat().st_mtime
            if mtime > best_mtime:
                best_mtime = mtime
                best = path
    return best


def _download_or_link_file(path: Path) -> None:
    resolved = path.resolve()
    env = _detect_notebook_env()
    if env == "colab":
        from google.colab import files

        files.download(str(resolved))
        return
    if env == "kaggle":
        from IPython.display import FileLink, display

        dest = Path("/kaggle/working") / resolved.name
        if resolved != dest.resolve():
            dest.write_bytes(resolved.read_bytes())
            print(f"Copied to {dest}")
        try:
            link_target = str(dest.relative_to(Path.cwd()))
        except ValueError:
            link_target = dest.name
        display(FileLink(link_target))
        print("Use the link above, or download from the Kaggle Output panel.")
        return
    print(f"File ready at: {resolved}")

REPO_DIR = _prepare_repo()
latest = _find_latest_podcast_mp3()
if latest:
    print(f"Downloading: {latest}")
    _download_or_link_file(latest)
else:
    print("No podcast file found.")
    print("cwd:", os.getcwd())
    print("Repo:", REPO_DIR)
    print("If you already generated one, the runtime may have wiped working storage.")
    print("Re-run cells 1, 2, and 4, or download your saved MP3 from Output manually.")
